# 15 — Actor Runtime Deep Dive

This notebook demonstrates the **actor-based agent runtime** — the messaging backbone of the framework.

### What you'll explore

| # | Section | Feature |
|---|---------|------|
| 1 | Identity types | `AgentId`, `TopicId` (validated, hashable, frozen) |
| 2 | Point-to-point messaging | `send_message()` — request/response with lazy agent creation |
| 3 | Pub/sub broadcast | `publish_message()` + `subscribe()` — fan-out to multiple agents |
| 4 | Streaming | `StreamPublisher` — ordered event streams with `StreamDone` sentinel |
| 5 | Supervisor | Erlang-style crash recovery (one-for-one, restart budget) |
| 6 | Cancellation | `CancellationToken` — cooperative cancellation of in-flight messages |
| 7 | Timeout & errors | `HandlerError`, configurable `send_timeout` |
| 8 | Multi-agent pipeline | Chaining agents via runtime — agent A calls agent B |
| 9 | Comparison with AutoGen | Side-by-side comparison |

### No external infrastructure required
Everything runs in-process using `LocalRuntime` (pure asyncio).

In [22]:
import asyncio
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

## 1. Identity Types — `AgentId` and `TopicId`

Every agent and topic is uniquely identified by a frozen, hashable dataclass.

- **`AgentId(type, key)`** — `type` is the agent class, `key` scopes the instance
- **`TopicId(type, source)`** — `type` is the event category, `source` scopes the topic

Both validate their fields on construction (alphanumeric, underscores, hyphens, dots).

In [23]:
from raavan.core.runtime import AgentId, TopicId

# Valid identifiers
agent = AgentId(type="react_agent", key="thread-abc-123")
topic = TopicId(type="sse_events", source="session.42")

print(f"Agent: {agent}")
print(f"Topic: {topic}")
print(f"Hashable: {hash(agent)}, {hash(topic)}")
print(f"Dict key: { {agent: 'works', topic: 'too'} }")

Agent: react_agent/thread-abc-123
Topic: sse_events/session.42
Hashable: -2539129451621234945, 6267636583204590668
Dict key: {AgentId(type='react_agent', key='thread-abc-123'): 'works', TopicId(type='sse_events', source='session.42'): 'too'}


In [24]:
# Invalid identifiers are rejected at construction time
try:
    AgentId(type="bad agent", key="1")  # spaces not allowed
except ValueError as e:
    print(f"✓ Rejected: {e}")

try:
    AgentId(type="agent", key="../../etc/passwd")  # path traversal attempt
except ValueError as e:
    print(f"✓ Rejected: {e}")

✓ Rejected: Invalid agent type: 'bad agent'. Must match [\w\-.]+.
✓ Rejected: Invalid agent key: '../../etc/passwd'. Must match [\w\-.]+.


## 2. Point-to-Point Messaging

The core primitive: **send a message to an agent, get a response back**.

### How it works:
1. Register a handler function for an agent type
2. Call `send_message()` with a recipient `AgentId`
3. The runtime **lazily creates** the agent on first message
4. The handler runs, and its return value becomes the response

In [25]:
from raavan.core.runtime import LocalRuntime, AgentId, MessageContext
from typing import Any

# Create runtime
runtime = LocalRuntime()
await runtime.start()

# Define a handler — receives (context, payload), returns a response
async def echo_handler(ctx: MessageContext, payload: Any) -> Any:
    print(f"  [echo/{ctx.agent_id.key}] received: {payload!r}")
    return f"Echo: {payload}"

# Register the handler for agent type "echo"
await runtime.register("echo", echo_handler)

# Send messages — agents are created lazily
r1 = await runtime.send_message(
    "Hello, world!",
    sender=AgentId("user", "u1"),
    recipient=AgentId("echo", "instance-1"),
)
print(f"Response: {r1}")

# Same type, different key = different agent instance
r2 = await runtime.send_message(
    "Different instance",
    sender=AgentId("user", "u1"),
    recipient=AgentId("echo", "instance-2"),
)
print(f"Response: {r2}")

# Introspect
print(f"\nRegistered types: {runtime.registered_types}")
print(f"Active agents: {runtime.active_agents}")

await runtime.stop()

LocalRuntime started
  [echo/instance-1] received: 'Hello, world!'
Response: Echo: Hello, world!
  [echo/instance-2] received: 'Different instance'
Response: Echo: Different instance

Registered types: ['echo']
Active agents: [AgentId(type='echo', key='instance-2'), AgentId(type='echo', key='instance-1')]
LocalRuntime stopped


## 3. Pub/Sub Broadcast

Publish a message to a `TopicId` — all subscribed agent types receive it.

Key behaviors:
- **Fan-out**: one publish → N subscribers
- **Fault-isolated**: if one subscriber fails, the others still receive the message
- **Lazy**: subscriber agents are created on first publish

In [26]:
runtime = LocalRuntime()
await runtime.start()

# Subscriber A — analytics
analytics_events = []
async def analytics_handler(ctx: MessageContext, payload: Any) -> Any:
    analytics_events.append(payload)
    print(f"  [analytics] logged: {payload}")
    return None

# Subscriber B — notifications
notifications = []
async def notification_handler(ctx: MessageContext, payload: Any) -> Any:
    notifications.append(payload)
    print(f"  [notifications] alert: {payload}")
    return None

# Register both
await runtime.register("analytics", analytics_handler)
await runtime.register("notifications", notification_handler)

# Subscribe both to the same topic
topic = TopicId(type="user_events", source="app")
await runtime.subscribe("analytics", topic)
await runtime.subscribe("notifications", topic)

# Publish — both subscribers receive it
await runtime.publish_message(
    {"event": "user_signed_up", "user_id": "u42"},
    sender=AgentId("auth_service", "main"),
    topic=topic,
)

# Small delay for async delivery
await asyncio.sleep(0.1)

print(f"\nAnalytics received: {len(analytics_events)} events")
print(f"Notifications received: {len(notifications)} events")

await runtime.stop()

LocalRuntime started
  [analytics] logged: {'event': 'user_signed_up', 'user_id': 'u42'}
  [notifications] alert: {'event': 'user_signed_up', 'user_id': 'u42'}

Analytics received: 1 events
Notifications received: 1 events
LocalRuntime stopped


## 4. Streaming with `StreamPublisher`

`StreamPublisher` is purpose-built for **ordered event streams** — like SSE chunks from an LLM.

- Wraps `publish_message()` with lifecycle management
- Sends a `StreamDone` sentinel when the stream ends
- Thread-safe: `asyncio.Lock` prevents emit/close races

In [27]:
from raavan.core.runtime import StreamPublisher, StreamDone, TopicId, AgentId

runtime = LocalRuntime()
await runtime.start()

# Collect stream events
stream_events = []

async def stream_consumer(ctx: MessageContext, payload: Any) -> Any:
    if isinstance(payload, StreamDone):
        print(f"  [consumer] stream ended: {payload.reason}")
    else:
        print(f"  [consumer] chunk: {payload}")
    stream_events.append(payload)
    return None

await runtime.register("consumer", stream_consumer)
stream_topic = TopicId(type="llm_stream", source="chat-1")
await runtime.subscribe("consumer", stream_topic)

# Create a stream publisher
publisher = StreamPublisher(
    runtime,
    stream_topic,
    sender=AgentId("llm", "gpt-4o"),
)

# Emit chunks
await publisher.emit({"type": "text_delta", "content": "Hello "})
await publisher.emit({"type": "text_delta", "content": "world!"})
await publisher.emit({"type": "tool_call", "name": "search", "input": {"q": "latest news"}})
await publisher.close()  # Sends StreamDone

await asyncio.sleep(0.1)

print(f"\nTotal events received: {len(stream_events)}")
print(f"Stream topic: {publisher.topic}")

# After close, emit raises
try:
    await publisher.emit({"more": "data"})
except RuntimeError as e:
    print(f"✓ Correctly rejected: {e}")

await runtime.stop()

LocalRuntime started
  [consumer] chunk: {'type': 'text_delta', 'content': 'Hello '}
  [consumer] chunk: {'type': 'text_delta', 'content': 'world!'}
  [consumer] chunk: {'type': 'tool_call', 'name': 'search', 'input': {'q': 'latest news'}}
  [consumer] stream ended: complete

Total events received: 4
Stream topic: llm_stream/chat-1
✓ Correctly rejected: StreamPublisher is closed
LocalRuntime stopped


## 5. Supervisor — Erlang-Style Crash Recovery

The `Supervisor` monitors agent tasks and restarts them on crash.

### Restart strategies:
- **`one_for_one`** — restart only the crashed agent
- **`one_for_all`** — restart all supervised agents

### Restart budget:
If an agent crashes more than `max_restarts` times within `restart_window` seconds,
the supervisor **escalates** (`SupervisorEscalation`) instead of restarting.

In [28]:
from raavan.core.runtime import Supervisor, RestartPolicy, SupervisorEscalation, AgentId

# Agent that crashes 2 times then recovers
crash_count = 0

async def flaky_agent():
    global crash_count
    crash_count += 1
    if crash_count <= 2:
        print(f"  [flaky] attempt #{crash_count} — crashing!")
        raise RuntimeError(f"crash #{crash_count}")
    print(f"  [flaky] attempt #{crash_count} — running OK!")
    await asyncio.sleep(5)  # stay alive

# Policy: allow up to 5 restarts in 60s
policy = RestartPolicy(max_restarts=5, restart_window=60.0)
supervisor = Supervisor(restart_policy=policy)

aid = AgentId("flaky", "1")
supervisor.supervise(aid, flaky_agent)

await asyncio.sleep(0.5)  # let crashes + restarts happen

print(f"\nTotal attempts: {crash_count}")
print(f"Agent still supervised: {aid in supervisor.supervised_agents}")

await supervisor.stop_all()

  [flaky] attempt #1 — crashing!
⚠ agent flaky/1 crashed: crash #1
restarted agent flaky/1 (attempt 1)
  [flaky] attempt #2 — crashing!
⚠ agent flaky/1 crashed: crash #2
restarted agent flaky/1 (attempt 2)
  [flaky] attempt #3 — running OK!

Total attempts: 3
Agent still supervised: True


In [29]:
# Budget exceeded → escalation
budget_crash_count = 0

async def always_crash():
    global budget_crash_count
    budget_crash_count += 1
    raise RuntimeError(f"crash #{budget_crash_count}")

# Policy: only 2 restarts allowed
strict_policy = RestartPolicy(max_restarts=2, restart_window=60.0)
strict_sup = Supervisor(restart_policy=strict_policy)

aid = AgentId("doomed", "1")
strict_sup.supervise(aid, always_crash)

await asyncio.sleep(0.5)

# The last task should have a SupervisorEscalation
final_task = strict_sup._tasks.get(aid)
if final_task and final_task.done():
    exc = final_task.exception()
    print(f"✓ Escalated: {type(exc).__name__}: {exc}")
print(f"Total crash attempts: {budget_crash_count}")

await strict_sup.stop_all()

⚠ agent doomed/1 crashed: crash #1
restarted agent doomed/1 (attempt 1)
⚠ agent doomed/1 crashed: crash #2
restarted agent doomed/1 (attempt 2)
⚠ agent doomed/1 crashed: crash #3
✖ agent doomed/1 exceeded restart budget (2 in 60s) — escalating
✓ Escalated: SupervisorEscalation: agent doomed/1 crashed 3 times within 60.0s
Total crash attempts: 3


## 6. CancellationToken

Cooperative cancellation for in-flight `send_message` calls.
Works like AutoGen's `CancellationToken` — cancel from outside while the handler is running.

In [30]:
from raavan.core.runtime import CancellationToken

runtime = LocalRuntime()
await runtime.start()

async def slow_handler(ctx: MessageContext, payload: Any) -> Any:
    print("  [slow] starting long computation...")
    await asyncio.sleep(10)  # will be cancelled
    return "done"  # never reached

await runtime.register("slow", slow_handler)

# Create a cancellation token
token = CancellationToken()

# Start a send_message in a task
task = asyncio.create_task(
    runtime.send_message(
        "long job",
        sender=AgentId("user", "u1"),
        recipient=AgentId("slow", "w1"),
        cancellation_token=token,
    )
)

# Cancel after a short delay
await asyncio.sleep(0.1)
print("Cancelling...")
token.cancel()

try:
    await task
except asyncio.CancelledError:
    print("✓ Send was cancelled via CancellationToken")

await runtime.stop()

LocalRuntime started
  [slow] starting long computation...
Cancelling...
✓ Send was cancelled via CancellationToken
LocalRuntime stopped


## 7. Timeout & Error Handling

- **`send_timeout`** — configurable timeout for `send_message()` (default: 30s)
- **`HandlerError`** — wraps handler exceptions so callers get proper errors instead of silent `None`

In [31]:
from raavan.core.runtime import HandlerError

# Runtime with a tight timeout
runtime = LocalRuntime(send_timeout=1.0)  # 1 second timeout
await runtime.start()

# Handler that crashes
async def bad_handler(ctx: MessageContext, payload: Any) -> Any:
    raise ValueError(f"invalid input: {payload}")

# Handler that's too slow
async def stalled_handler(ctx: MessageContext, payload: Any) -> Any:
    await asyncio.sleep(999)
    return "never"

await runtime.register("bad", bad_handler)
await runtime.register("stalled", stalled_handler)

# Test 1: handler crash → HandlerError
try:
    await runtime.send_message(
        "boom",
        sender=AgentId("user", "u1"),
        recipient=AgentId("bad", "1"),
    )
except HandlerError as e:
    print(f"✓ HandlerError: {e}")

# Test 2: timeout
try:
    await runtime.send_message(
        "hurry",
        sender=AgentId("user", "u1"),
        recipient=AgentId("stalled", "1"),
    )
except TimeoutError as e:
    print(f"✓ TimeoutError: {e}")

await runtime.stop()

LocalRuntime started
✖ handler for bad/1 raised on message fd7f0f0b3d884e09bb4f906fffef168d
✓ HandlerError: handler for bad/1 raised: invalid input: boom
✓ TimeoutError: send_message to stalled/1 timed out after 1.0s
LocalRuntime stopped


## 8. Multi-Agent Pipeline

Agents can call other agents through the `MessageContext.runtime` reference.
This enables **agent chaining** — like a pipeline where each stage delegates to the next.

In [32]:
runtime = LocalRuntime()
await runtime.start()

# Stage 1: Preprocessor — normalizes input
async def preprocessor(ctx: MessageContext, payload: Any) -> Any:
    text = payload["text"].strip().lower()
    print(f"  [preprocessor] normalized: {text!r}")
    
    # Forward to analyzer via runtime
    result = await ctx.runtime.send_message(
        {"text": text, "original": payload["text"]},
        sender=ctx.agent_id,
        recipient=AgentId("analyzer", ctx.agent_id.key),
    )
    return result

# Stage 2: Analyzer — processes the text
async def analyzer(ctx: MessageContext, payload: Any) -> Any:
    text = payload["text"]
    analysis = {
        "word_count": len(text.split()),
        "char_count": len(text),
        "original": payload["original"],
        "normalized": text,
    }
    print(f"  [analyzer] result: {analysis}")
    return analysis

await runtime.register("preprocessor", preprocessor)
await runtime.register("analyzer", analyzer)

# Send to preprocessor — it chains to analyzer automatically
result = await runtime.send_message(
    {"text": "  Hello Beautiful World!  "},
    sender=AgentId("user", "u1"),
    recipient=AgentId("preprocessor", "pipeline-1"),
)

print(f"\nFinal result: {result}")

await runtime.stop()

LocalRuntime started
  [preprocessor] normalized: 'hello beautiful world!'
  [analyzer] result: {'word_count': 3, 'char_count': 22, 'original': '  Hello Beautiful World!  ', 'normalized': 'hello beautiful world!'}

Final result: {'word_count': 3, 'char_count': 22, 'original': '  Hello Beautiful World!  ', 'normalized': 'hello beautiful world!'}
LocalRuntime stopped


## 9. Full Example — Chat Simulation with SSE-like Streaming

This combines multiple runtime features into a realistic scenario:
- A **chat agent** processes user messages
- A **logger** subscribes to all events via pub/sub
- Responses are streamed chunk-by-chunk via `StreamPublisher`

In [33]:
runtime = LocalRuntime()
await runtime.start()

# Simulated LLM response chunks very long
FAKE_RESPONSE_LINES = [
["Hel","lo"," ","thi","s ","is ","a ","str","eam","ing ","res","pon","se ","for ","test","ing ","you","r ","age","nt",". ","It ","sim","ula","tes ","rea","l ","tok","en ","flo","w ","beh","avi","or",". ","Ple","ase ","pro","ces","s ","thi","s ","cor","rec","tly","."],
["We ","are ","gen","era","ting ","mul","tip","le ","tok","ens ","to ","mim","ic ","LL","M ","out","put",". ","Eac","h ","chu","nk ","is ","sma","ll ","and ","inc","rem","ent","al",". ","Mak","e ","sur","e ","you","r ","pip","eli","ne ","han","dle","s ","thi","s ","wel","l","."],
["Som","e ","tok","ens ","may ","arr","ive ","spl","it ","mid","-wo","rd ","whi","ch ","is ","nor","mal ","in ","str","eam","ing",". ","Han","dle ","buf","fer","ing ","pro","per","ly ","to ","rec","ons","tru","ct ","the ","mes","sag","e",". "],
["Tes","tin","g ","err","or ","han","dli","ng ","is ","imp","ort","ant",". ","Som","eti","mes ","tok","ens ","may ","be ","mis","sin","g ","or ","del","aye","d",". ","Ens","ure ","you","r ","sys","tem ","is ","res","ili","ent",". "],
["We ","als","o ","inc","lude ","pun","ctu","ati","on ","spl","its ",", ","lik","e ","thi","s ",", ","to ","see ","if ","you ","mer","ge ","the","m ","cor","rec","tly",". ","Thi","s ","is ","use","ful ","for ","UI ","ren","der","ing","."],
["Ano","the","r ","lin","e ","wit","h ","dif","fer","ent ","pat","ter","ns ","and ","spa","cin","g ","to ","ens","ure ","rob","ust","nes","s ","in ","str","eam","ing ","par","ser","s",". ","Kee","p ","tes","tin","g ","tho","rou","ghl","y","."],
["You","r ","age","nt ","sho","uld ","col","lec","t ","chu","nks ","and ","bui","ld ","the ","fin","al ","out","put ","gra","dua","lly",". ","Do ","not ","wai","t ","for ","ful","l ","res","pon","se ","to ","arr","ive",". "],
["Som","e ","lin","es ","wil","l ","hav","e ","ext","ra ","spa","ces ","or ","odd ","spl","its ","to ","sim","ula","te ","rea","l ","wor","ld ","noi","se",". ","Mak","e ","sur","e ","you","r ","han","dli","ng ","is ","cor","rec","t","."],
["Thi","s ","dat","ase","t ","is ","use","ful ","for ","NAT","S ","or ","SSE ","str","eam","ing ","tes","tin","g",". ","It ","hel","ps ","you ","ver","ify ","ord","er","ing ","and ","con","sis","ten","cy",". "],
["You ","can ","als","o ","int","rod","uce ","del","ays ","bet","wee","n ","tok","ens ","to ","sim","ula","te ","net","wor","k ","lat","enc","y",". ","Thi","s ","wil","l ","rev","eal ","tim","ing ","iss","ues",". "],

["Dat","a ","pip","eli","nes ","mus","t ","be ","res","ili","ent ","to ","par","tia","l ","inp","uts",". ","You ","sho","uld ","log ","chu","nks ","and ","deb","ug ","acc","ord","ing","ly",". "],
["Som","eti","mes ","tok","ens ","arr","ive ","out ","of ","ord","er",". ","Tes","t ","you","r ","sys","tem ","for ","ord","er","ing ","gua","ran","tee","s",". "],
["Han","dle ","emp","ty ","tok","ens ","as ","wel","l",". ","The","y ","may ","app","ear ","in ","rea","l ","str","eam","ing ","sce","nar","ios",". "],
["Mak","e ","sur","e ","you","r ","buf","fer ","doe","s ","not ","gro","w ","unc","ont","rol","lab","ly",". ","Cle","an ","it ","per","iod","ica","lly",". "],
["Tes","t ","wit","h ","lon","g ","res","pon","ses ","as ","wel","l ","as ","sho","rt ","one","s",". ","Cov","er ","all ","edg","e ","cas","es",". "],
["You","r ","UI ","sho","uld ","ren","der ","tex","t ","pro","gre","ssi","vel","y",". ","Use","rs ","exp","ect ","rea","l ","tim","e ","upd","ate","s",". "],
["Sim","ula","te ","net","wor","k ","fai","lur","es ","and ","ret","rie","s ","to ","ens","ure ","rel","iab","ili","ty",". "],
["Thi","s ","wil","l ","hel","p ","you ","bui","ld ","pro","duc","tio","n ","gra","de ","sys","tem","s",". "],
["You ","can ","ext","end ","thi","s ","dat","ase","t ","fur","the","r ","if ","nee","ded",". "],
["Tes","t ","agg","res","siv","ely ","bef","ore ","dep","loy","men","t",". "],

# (Continuing similar structure up to ~100 lines)

["Mor","e ","lin","es ","fol","low ","wit","h ","ran","dom ","tok","ens ","for ","rob","ust ","tes","tin","g",". ","Ens","ure ","sca","lab","ili","ty",". "],
["Str","eam","ing ","sys","tem","s ","mus","t ","be ","fas","t ","and ","eff","ici","ent",". ","Opt","imi","ze ","lat","enc","y",". "],
["Tes","t ","mem","ory ","usa","ge ","dur","ing ","str","eam","ing ","ope","rat","ion","s",". "],
["Ens","ure ","thr","ead","-sa","fet","y ","if ","app","lic","abl","e",". "],
["Val","ida","te ","out","put ","cor","rec","tne","ss ","aft","er ","rec","ons","tru","cti","on",". "],
["Use"," ","log","gin","g ","to ","tra","ck ","tok","en ","flo","w",". "],
["Deb","ug ","any ","mis","mat","che","s ","in ","out","put",". "],
["Han","dle ","uni","cod","e ","and ","spe","cia","l ","cha","rac","ter","s",". "],
["Ens","ure ","com","pat","ibi","lit","y ","acr","oss ","env","iro","nme","nts",". "],
["Thi","s ","is ","a ","com","pre","hen","siv","e ","tes","t ","dat","ase","t",". "],

# Fill remaining lines similarly to reach ~100 total
]
collected_chunks = []

# Chat agent — streams response via StreamPublisher
async def chat_agent(ctx: MessageContext, payload: Any) -> Any:
    user_msg = payload["message"]
    stream_topic = TopicId(type="chat_stream", source=ctx.agent_id.key)
    
    publisher = StreamPublisher(ctx.runtime, stream_topic, sender=ctx.agent_id)
    
    for chunk in FAKE_RESPONSE:
        await publisher.emit({"type": "text_delta", "content": chunk})
        await asyncio.sleep(0.05)  # simulate LLM latency
    
    await publisher.close()
    return {"status": "complete", "chunks": len(FAKE_RESPONSE)}

# Stream consumer — collects SSE-like events
async def stream_handler(ctx: MessageContext, payload: Any) -> Any:
    if isinstance(payload, StreamDone):
        collected_chunks.append("[DONE]")
    else:
        collected_chunks.append(payload["content"])
    return None

# Event logger — subscribes to audit log
audit_log = []
async def audit_handler(ctx: MessageContext, payload: Any) -> Any:
    audit_log.append({"from": str(ctx.sender), "payload_type": type(payload).__name__})
    return None

# Register all agents
await runtime.register("chat_agent", chat_agent)
await runtime.register("stream_consumer", stream_handler)
await runtime.register("auditor", audit_handler)

# Subscribe consumers to stream topic
chat_topic = TopicId(type="chat_stream", source="thread-1")
await runtime.subscribe("stream_consumer", chat_topic)
await runtime.subscribe("auditor", chat_topic)

# Send user message to chat agent
result = await runtime.send_message(
    {"message": "What can you do?"},
    sender=AgentId("user", "u1"),
    recipient=AgentId("chat_agent", "thread-1"),
)

await asyncio.sleep(0.3)  # let pub/sub deliver

print(f"Chat result: {result}")
print(f"\nStreamed text: {''.join(c for c in collected_chunks if c != '[DONE]')}")
print(f"Chunks received: {len(collected_chunks)} (including DONE sentinel)")
print(f"Audit log entries: {len(audit_log)}")

await runtime.stop()

LocalRuntime started


Chat result: {'status': 'complete', 'chunks': 29}

Streamed text: Hello! I'm an AI assistant designed to help with a wide range of tasks. I can assist you with data analysis, software development, system design, and even help you debug complex issues in your workflows. If you're working on something like distributed systems or agent frameworks, I can provide architectural suggestions, best practices, and performance optimizations. For example, if you're building a pipeline, I can guide you on orchestration tools, data modeling, and scaling strategies. I can also help with writing clean and maintainable code, including Python, SQL, and other technologies you're using. If you need help understanding errors, just share the logs and I’ll break them down step by step. I can also generate summaries, extract structured information from text, and assist with natural language processing tasks. In addition, I can help you brainstorm product ideas, especially if you're trying to build something s

## 10. `stop_when_idle()` — Wait for All Work to Finish

Instead of `stop()` (immediate), `stop_when_idle()` waits until all mailboxes are empty
and all pending responses are resolved — then shuts down cleanly.

In [34]:
runtime = LocalRuntime()
await runtime.start()

processed = []

async def batch_worker(ctx: MessageContext, payload: Any) -> Any:
    await asyncio.sleep(0.1)  # simulate work
    processed.append(payload)
    return None

await runtime.register("worker", batch_worker)
topic = TopicId(type="batch", source="job-1")
await runtime.subscribe("worker", topic)

# Fire off many pub/sub messages
for i in range(5):
    await runtime.publish_message(
        {"item": i},
        sender=AgentId("dispatcher", "main"),
        topic=topic,
    )

# Give the event loop a tick so dispatched messages land in mailboxes
await asyncio.sleep(0.05)

# Wait for all work to finish before stopping
await runtime.stop_when_idle()
print(f"✓ Processed {len(processed)} items before shutdown")
print(f"Items: {processed}")

LocalRuntime started
LocalRuntime stopped
✓ Processed 5 items before shutdown
Items: [{'item': 0}, {'item': 1}, {'item': 2}, {'item': 3}, {'item': 4}]


## 11. Low-Level Building Blocks

The runtime is composed of reusable primitives you can use independently.

In [35]:
from raavan.core.runtime import Mailbox, MailboxFullError, Envelope, Dispatcher, AgentNotFoundError

# === Mailbox: bounded async queue with close protocol ===
mbox = Mailbox(capacity=3)

# Put messages
for i in range(3):
    envelope = Envelope(sender=None, target=AgentId("test", "1"), payload=f"msg-{i}")
    await mbox.put(envelope)

print(f"Mailbox size: {mbox.size}, full: {mbox.is_full}")

# Full → raises
try:
    mbox.put_nowait(Envelope(sender=None, target=AgentId("test", "1"), payload="overflow"))
except MailboxFullError as e:
    print(f"✓ Backpressure: {e}")

# Consume
env = await mbox.get()
print(f"Got: {env.payload}")
print(f"Mailbox size after get: {mbox.size}")

mbox.close()
print(f"Closed: {mbox.closed}")

Mailbox size: 3, full: True
✓ Backpressure: mailbox at capacity (3)
Got: msg-0
Mailbox size after get: 2
Closed: True


In [36]:
# === Dispatcher: routing table ===
dispatcher = Dispatcher()

# Register 2 agents
mbox_a = Mailbox(capacity=10)
mbox_b = Mailbox(capacity=10)
aid_a = AgentId("worker", "a")
aid_b = AgentId("worker", "b")

dispatcher.register_agent(aid_a, mbox_a)
dispatcher.register_agent(aid_b, mbox_b)

# Subscribe to topic
topic = TopicId(type="tasks", source="queue")
dispatcher.subscribe_to_topic(topic, "worker")

# Direct dispatch
await dispatcher.dispatch(Envelope(sender=None, target=aid_a, payload="direct-to-a"))
print(f"worker/a mailbox: {mbox_a.size}")

# Topic fan-out → both workers receive
await dispatcher.dispatch(Envelope(sender=None, target=topic, payload="broadcast"))
print(f"worker/a mailbox after broadcast: {mbox_a.size}")
print(f"worker/b mailbox after broadcast: {mbox_b.size}")

# Unknown agent → error    
try:
    await dispatcher.dispatch(Envelope(sender=None, target=AgentId("unknown", "x"), payload="?"))
except AgentNotFoundError as e:
    print(f"✓ {e}")

print(f"\nRegistered agents: {dispatcher.registered_agents}")
print(f"Topics: {dispatcher.topics}")

worker/a mailbox: 1
worker/a mailbox after broadcast: 2
worker/b mailbox after broadcast: 1
✓ no mailbox registered for unknown/x

Registered agents: [AgentId(type='worker', key='a'), AgentId(type='worker', key='b')]
Topics: [TopicId(type='tasks', source='queue')]


## 12. Comparison with AutoGen Runtime

| Feature | AutoGen `SingleThreadedAgentRuntime` | Raavan `LocalRuntime` |
|---------|--------------------------------------|----------------------|
| **Message queue** | Single global `asyncio.Queue` (unbounded) | Per-agent `Mailbox` (bounded) ✅ |
| **Backpressure** | ❌ No | ✅ `MailboxFullError` when at capacity |
| **Agent instantiation** | Lazy (on first message) | Lazy (on first message) |
| **Crash recovery** | ❌ None (stores background exception) | ✅ Erlang-style `Supervisor` |
| **Restart strategies** | ❌ None | ✅ `one_for_one`, `one_for_all` |
| **Restart budget** | ❌ None | ✅ `max_restarts` within `restart_window` |
| **CancellationToken** | ✅ Yes | ✅ Yes |
| **Message timeout** | ❌ No (waits forever) | ✅ Configurable `send_timeout` |
| **Error propagation** | ✅ Via future exception | ✅ `HandlerError` wrapping |
| **Fan-out isolation** | ❌ `asyncio.gather` — one failure propagates | ✅ Per-subscriber try/except |
| **Streaming** | ❌ No built-in | ✅ `StreamPublisher` + `StreamDone` |
| **InterventionHandler** | ✅ Intercept send/publish/response | ❌ Not yet |
| **State save/load** | ✅ `save_state()` / `load_state()` | ❌ Not yet |
| **Message serialization** | ✅ `SerializationRegistry` | ❌ Uses `Any` (simpler) |
| **OpenTelemetry** | ✅ Per-message tracing | ✅ Higher-level (via framework) |
| **`stop_when_idle()`** | ✅ Yes | ✅ Yes |
| **ID validation** | ✅ Regex on AgentId.type | ✅ Regex on type + key |
| **Remote backends** | gRPC only | gRPC + Restate + NATS ✅ |
| **Zero-dep core** | ❌ Requires opentelemetry, protobuf | ✅ Pure Python |
| **Protocol-first** | Class hierarchy (`Agent` base class) | ✅ `@runtime_checkable Protocol` |

### Where we're stronger:
- **Crash recovery** — Erlang-style supervision
- **Backpressure** — bounded mailboxes prevent OOM
- **Fan-out isolation** — one subscriber failure doesn't crash others
- **Streaming** — first-class `StreamPublisher` for SSE-like patterns
- **Zero dependencies** — core runtime is pure Python

### Where AutoGen is ahead (and we may add later):
- **InterventionHandler** — middleware to intercept/modify/drop messages
- **State save/load** — snapshot and restore agent state
- **Message serialization** — type-safe message serialization registry

## Summary

The actor runtime provides a clean, production-grade messaging backbone:

```
┌──────────────────────────────────────────────────┐
│                  AgentRuntime Protocol            │
│  send_message() · publish_message() · register() │
│  subscribe()    · start()           · stop()     │
├──────────────────────────────────────────────────┤
│         LocalRuntime (in-process, asyncio)       │
│  ┌──────────┐  ┌──────────┐  ┌──────────────┐   │
│  │ Dispatcher│  │Supervisor│  │ StreamPublish│   │
│  │ (routing) │  │ (crash   │  │ er (ordered  │   │
│  │           │  │ recovery)│  │ events)      │   │
│  └──────────┘  └──────────┘  └──────────────┘   │
│       ▲              ▲                           │
│  ┌────┴──────────────┴──────────┐               │
│  │  Mailbox (bounded, async,    │               │
│  │  backpressure, close proto)  │               │
│  └─────────────────────────────┘                │
├──────────────────────────────────────────────────┤
│              Remote Backends                     │
│  GrpcRuntime · RestateRuntime · NATSBridge       │
└──────────────────────────────────────────────────┘
```